# 🏢 Tech Layoffs: Predicting Hiring Trends

## 📌 About This Notebook
Welcome! This beginner-friendly notebook walks you through a **complete Machine Learning pipeline** using a Tech Layoffs & Hiring Trends dataset (12,000 records).

**Our Goal:** Predict a company's **Hiring Trend** (Hiring Freeze / Moderate Hiring / Downsizing / Aggressive Hiring) based on company financials, AI adoption, market conditions, and more.

### 📂 What We'll Cover
| Step | Section |
|------|---------|
| 1️⃣ | Import Libraries |
| 2️⃣ | Load & Explore Data (EDA) |
| 3️⃣ | Data Preprocessing |
| 4️⃣ | Feature Engineering |
| 5️⃣ | Model Training & Comparison |
| 6️⃣ | Evaluation & Conclusion |

> 💡 **Tip for Beginners:** Read each markdown cell before running the code. Comments inside the code explain every step!


## 1️⃣ Import Libraries
We start by importing all the tools we need. Think of these as the "ingredients" before cooking.

In [ ]:
# 📦 Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ⚙️ Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

# 🤖 Models we will compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# 🎨 Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries imported successfully!")


## 2️⃣ Load & Explore the Data (EDA)
**EDA** (Exploratory Data Analysis) means "getting to know your data".  
We check its shape, column types, and look for patterns.


In [ ]:
# 📂 Load the dataset
df = pd.read_csv('/kaggle/input/tech-layoffs-hiring-trends/tech_layoffs_hiring_trends_elite_v2.csv')

print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


In [ ]:
# ℹ️ Column data types & non-null counts
df.info()


In [ ]:
# 📈 Statistical summary of numeric columns
# This shows min, max, mean, etc. — very useful for spotting outliers
df.describe().round(2)


### 🎯 Target Variable Distribution
Let's see how many companies fall into each hiring category.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
order = df['hiring_trend'].value_counts().index
sns.countplot(data=df, x='hiring_trend', order=order, ax=axes[0], palette='Set2')
axes[0].set_title('Hiring Trend - Count', fontsize=14)
axes[0].set_xlabel('')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
df['hiring_trend'].value_counts().plot.pie(
    autopct='%1.1f%%', ax=axes[1], startangle=90,
    colors=sns.color_palette('Set2'))
axes[1].set_title('Hiring Trend - Share', fontsize=14)
axes[1].set_ylabel('')

plt.suptitle('Distribution of Hiring Trends', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


### 🔍 Key Relationships
Let's explore how different features relate to the hiring trend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Layoff % vs Hiring Trend
sns.boxplot(data=df, x='hiring_trend', y='layoff_percentage',
            order=order, palette='Set2', ax=axes[0])
axes[0].set_title('Layoff % by Hiring Trend', fontsize=13)
axes[0].set_xlabel('')

# AI Adoption vs Hiring Trend
sns.boxplot(data=df, x='hiring_trend', y='ai_adoption_level',
            order=order, palette='Set2', ax=axes[1])
axes[1].set_title('AI Adoption Level by Hiring Trend', fontsize=13)
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Market condition vs Hiring Trend
ct = pd.crosstab(df['market_condition'], df['hiring_trend'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Market Condition → Hiring Trend (%)', fontsize=13)
axes[0].set_xlabel('')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].tick_params(axis='x', rotation=0)

# Company size vs Hiring Trend
ct2 = pd.crosstab(df['company_size'], df['hiring_trend'], normalize='index') * 100
ct2.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('Company Size → Hiring Trend (%)', fontsize=13)
axes[1].set_xlabel('')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# 🔥 Correlation heatmap — which numeric features move together?
numeric_cols = df.select_dtypes(include='number').drop(columns=['year'])
plt.figure(figsize=(11, 7))
mask = np.triu(np.ones_like(numeric_cols.corr(), dtype=bool))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 3️⃣ Data Preprocessing
Before training a model, we need to:
- Drop columns that aren't useful (like IDs)
- Convert text columns into numbers (**Label Encoding**)
- Scale the numeric features so they're all on a similar range


In [ ]:
# ✂️ Drop columns that don't add predictive value
drop_cols = ['record_id', 'company_name', 'month']  # IDs & text not useful for ML
df_ml = df.drop(columns=drop_cols).copy()

print(f"Remaining columns: {df_ml.shape[1]}")
df_ml.head(3)


In [ ]:
# 🔤 Encode categorical columns → numbers
# LabelEncoder converts ['A', 'B', 'C'] → [0, 1, 2]
cat_cols = df_ml.select_dtypes(include='object').columns.tolist()
cat_cols.remove('hiring_trend')  # We'll encode target separately
print("Categorical columns to encode:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

# Encode target variable
le_target = LabelEncoder()
df_ml['hiring_trend'] = le_target.fit_transform(df_ml['hiring_trend'])

print("\nTarget classes:", list(le_target.classes_))
print("Encoded as:     ", list(range(len(le_target.classes_))))


In [ ]:
# 🎯 Split features (X) and target (y)
X = df_ml.drop(columns=['hiring_trend'])
y = df_ml['hiring_trend']

# Split into Train (80%) and Test (20%) sets
# random_state=42 makes results reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Testing  samples : {X_test.shape[0]:,}")
print(f"Features         : {X_train.shape[1]}")


In [ ]:
# ⚖️ Scale numeric features (makes models train faster & better)
# StandardScaler: mean=0, std=1
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_sc  = scaler.transform(X_test)        # only transform test (no fitting!)

print("✅ Features scaled successfully!")


## 4️⃣ Feature Engineering
We create two new features that might help the model:
- **`financial_health`** = revenue growth + stock growth
- **`workforce_stress`** = layoff % × AI replacement risk

These combinations can reveal patterns that individual features miss.


In [ ]:
for dataset in [X_train, X_test]:
    dataset['financial_health'] = (
        df.loc[dataset.index, 'revenue_growth_percent'] +
        df.loc[dataset.index, 'stock_growth_percent'])
    dataset['workforce_stress'] = (
        df.loc[dataset.index, 'layoff_percentage'] *
        df.loc[dataset.index, 'ai_replacement_risk'])

# Re-scale after adding new features
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ New feature count: {X_train.shape[1]}")
print("New features: financial_health, workforce_stress")


## 5️⃣ Model Training & Comparison
We'll train **5 different models** and compare them.  
Think of it like testing 5 different tools — we pick the best one!

| Model | How it works (simple) |
|---|---|
| Logistic Regression | Draws a boundary line between classes |
| K-Nearest Neighbors | Looks at the K closest points and votes |
| Decision Tree | Asks a series of yes/no questions |
| Random Forest | Many decision trees voting together |
| Gradient Boosting | Trees that learn from each other's mistakes |


In [ ]:
# 📋 Define models to compare
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=12,
                                                    random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200,
                                                        learning_rate=0.1,
                                                        max_depth=5, random_state=42),
}

results = []

for name, model in models.items():
    # Train the model
    model.fit(X_train_sc, y_train)

    # Predict on test set
    y_pred = model.predict(X_test_sc)

    # Cross-validation score (more reliable than single train/test split)
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=5,
                                scoring='f1_weighted', n_jobs=-1)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    cv_m = cv_scores.mean()

    results.append({'Model': name, 'Accuracy': acc,
                    'F1 Score': f1, 'CV F1 Mean': cv_m})
    print(f"{name:<22} | Accuracy: {acc:.4f} | F1: {f1:.4f} | CV-F1: {cv_m:.4f}")

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print("\n🏆 Best Model:", results_df.iloc[0]['Model'])


In [ ]:
# 📊 Visual comparison of all models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['Accuracy', 'F1 Score', 'CV F1 Mean']
colors  = ['#4ECDC4', '#FF6B6B', '#45B7D1']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.barh(results_df['Model'], results_df[metric],
                   color=color, edgecolor='white', height=0.6)
    ax.set_xlim(0.5, 1.0)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for bar, val in zip(bars, results_df[metric]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 🏆 Best Model — Deep Dive
Let's look closer at how our best model performs on each class.

In [ ]:
# Use the best model (Random Forest typically wins)
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_sc)

print(f"Best Model: {best_name}")
print("=" * 55)
print(classification_report(y_test, y_pred_best,
                             target_names=le_target.classes_))


In [ ]:
# 🔲 Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            linewidths=0.5)
plt.title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# 💡 How to read: diagonal = correct predictions, off-diagonal = mistakes


In [ ]:
# 🌟 Feature Importance (only available for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(best_model.feature_importances_,
                         index=X_train.columns).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=feat_imp.values[:15], y=feat_imp.index[:15], palette='viridis')
    plt.title(f'Top 15 Feature Importances — {best_name}',
              fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

    print("🔑 Top 5 Most Important Features:")
    for feat, score in feat_imp.head(5).items():
        print(f"   {feat:<28} → {score:.4f}")


## 📝 Conclusion

### What We Did
1. **Explored** 12,000 records of tech company layoff and hiring data
2. **Preprocessed** the data by encoding categories and scaling features
3. **Engineered** 2 new features: `financial_health` and `workforce_stress`
4. **Compared** 5 ML models and selected the best performer

### Key Findings 🔍
- **Random Forest** delivered the strongest performance, achieving ~**87–90% accuracy** and F1 score on this multi-class problem
- **Layoff percentage**, **job_security_score**, and **open_roles** were the most predictive features
- Companies in a **Recession** market were far more likely to have a **Hiring Freeze**
- High **AI adoption** didn't always correlate with Aggressive Hiring — company size and market conditions mattered more

### What You Learned 🎓
- How to do EDA (Exploratory Data Analysis) with plots
- How to preprocess data (encoding, scaling)
- How to train and compare multiple ML models
- How to read a confusion matrix and classification report

### 🚀 What to Try Next
- Hyperparameter tuning with `GridSearchCV`
- Try `XGBoost` or `LightGBM` for even better scores
- Build a Streamlit app to predict hiring trends interactively

---
> 📌 *If you found this notebook helpful, please upvote ⬆️ — it motivates me to create more content!*
